# 🎯 Direct Preference Optimization (DPO) Training

## What is DPO?
DPO aligns a language model with human preferences **without needing a reward model or reinforcement learning**.

### Old Way: RLHF (3 stages, complex)
```
Preference Data → Train Reward Model → Run PPO → Aligned Model
```

### New Way: DPO (1 stage, simple)
```
Preference Data → Train Directly → Aligned Model
```

### The Loss Function
```
loss = -log(sigmoid(β · [log(P/P_ref)_chosen - log(P/P_ref)_rejected]))
```

**In English:** "Make the model prefer chosen responses over rejected ones, relative to the original model."

- **β (beta)** = how conservative the alignment is (higher = stay closer to original)
- **P** = current model probability
- **P_ref** = reference (original) model probability

---
**Requirements:** GPU runtime (Runtime → Change runtime type → T4 GPU)

## Step 1: Install Dependencies & Check GPU

In [ ]:
!pip install -q torch transformers trl datasets peft accelerate bitsandbytes

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory: %.1f GB" % (torch.cuda.get_device_properties(0).total_mem / 1e9))
else:
    print("No GPU found! Go to Runtime > Change runtime type > T4 GPU")

## Step 2: Load the Model

We use **TinyLlama 1.1B** — small enough to train on a free T4 GPU, but big enough to show real DPO behavior.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print("Model loaded:", MODEL_NAME)
print("Parameters: %dM" % total_params)

## Step 3: Generate a "Before DPO" Response

Let's see what the model says **before** alignment so we can compare later.

In [ ]:
def generate(model, prompt, max_tokens=150):
    """Helper to generate a response from the model."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Test prompts we'll use before AND after DPO
TEST_PROMPTS = [
    "What is gravity?",
    "How do I learn Python?",
    "What makes a good friend?",
]

print("=" * 60)
print("BEFORE DPO (baseline responses)")
print("=" * 60)
before_responses = {}
for p in TEST_PROMPTS:
    resp = generate(model, p)
    before_responses[p] = resp
    print(f"\n📝 Prompt: {p}")
    print(f"💬 Response: {resp}")

## Step 4: Create Preference Data

This is the heart of DPO. Each example has:
- **prompt** — the user question
- **chosen** — the response humans preferred (detailed, helpful)
- **rejected** — the response humans didn't prefer (vague, lazy)

In production you'd use datasets like `Anthropic/hh-rlhf` or `argilla/ultrafeedback`. Here we use 5 hand-crafted examples to keep it dead simple.

In [ ]:
from datasets import Dataset

PREFERENCE_DATA = [
    {
        "prompt": "Explain what a black hole is.",
        "chosen": "A black hole is a region in space where gravity is so strong that nothing, not even light, can escape. They form when massive stars collapse at the end of their life cycle.",
        "rejected": "A black hole is a hole that is black. It sucks things in. Nobody really knows what they are.",
    },
    {
        "prompt": "How do I make scrambled eggs?",
        "chosen": "Crack 2-3 eggs into a bowl, whisk with salt and pepper. Heat butter in a pan over medium-low heat, pour in eggs, and gently stir with a spatula until softly set.",
        "rejected": "Put eggs in pan. Cook them. Add stuff if you want.",
    },
    {
        "prompt": "What is machine learning?",
        "chosen": "Machine learning is a branch of AI where computers learn patterns from data instead of being explicitly programmed. For example, a spam filter learns from labeled emails.",
        "rejected": "Machine learning is when computers learn stuff. It's really complicated and uses lots of math.",
    },
    {
        "prompt": "Why is exercise important?",
        "chosen": "Regular exercise strengthens your heart, improves mood by releasing endorphins, helps maintain a healthy weight, and reduces the risk of chronic diseases like diabetes.",
        "rejected": "Exercise is good for you. You should do it because everyone says so.",
    },
    {
        "prompt": "Explain recursion in programming.",
        "chosen": "Recursion is when a function calls itself to solve smaller sub-problems. For example, factorial(5) = 5 * factorial(4). Every recursive function needs a base case to stop.",
        "rejected": "Recursion is a hard concept. It's when things repeat. You'll understand it eventually.",
    },
]

def format_chat(prompt, response):
    return [{"role": "user", "content": prompt}, {"role": "assistant", "content": response}]

dataset = Dataset.from_list([
    {
        "prompt":   [{"role": "user", "content": ex["prompt"]}],
        "chosen":   format_chat(ex["prompt"], ex["chosen"]),
        "rejected": format_chat(ex["prompt"], ex["rejected"]),
    }
    for ex in PREFERENCE_DATA
])

print(f"✅ Dataset: {len(dataset)} preference pairs")
print(f"\nExample:")
print(f"  Prompt:   {PREFERENCE_DATA[0]['prompt']}")
print(f"  Chosen:   {PREFERENCE_DATA[0]['chosen'][:80]}...")
print(f"  Rejected: {PREFERENCE_DATA[0]['rejected'][:80]}...")

## Step 5: Configure LoRA + DPO Training

**LoRA** = Low-Rank Adaptation. Instead of fine-tuning all 1.1B parameters, we add tiny trainable adapters (~1% of params). This makes training fast and memory-efficient.

**Key DPO parameter:** `beta=0.1`
- Lower beta → more aggressive alignment (model changes more)
- Higher beta → conservative (model stays close to original)

In [ ]:
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

# LoRA config — small adapters on attention layers
lora_config = LoraConfig(
    r=16,                                  # adapter rank
    lora_alpha=32,                         # scaling factor
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],   # which layers to adapt
    bias="none",
    task_type="CAUSAL_LM",
)

# DPO training config
training_args = DPOConfig(
    output_dir="./dpo_output",
    beta=0.1,                              # ← THE key DPO hyperparameter
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=10,
    logging_steps=1,
    fp16=True,
    gradient_checkpointing=True,
    do_eval=False,
    remove_unused_columns=False,
    report_to="none",                      # no wandb needed
)

print(f"✅ Config ready")
print(f"   beta={training_args.beta}")
print(f"   epochs={int(training_args.num_train_epochs)}")
print(f"   lr={training_args.learning_rate}")
print(f"   LoRA rank={lora_config.r}")

## Step 6: Train! 🚀

This is where DPO happens. The trainer:
1. Freezes a copy of the model as the "reference" model
2. For each preference pair, computes log-probabilities for chosen & rejected under both models
3. Applies the DPO loss to push the model toward preferring chosen responses

In [ ]:
import json, time
from transformers import TrainerCallback

# Collect metrics for visualization
class MetricsLogger(TrainerCallback):
    def __init__(self):
        self.logs = []
        self.start = time.time()
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            entry = dict((k, v) for k, v in logs.items() if isinstance(v, (int, float)))
            entry["step"] = state.global_step
            entry["elapsed"] = round(time.time() - self.start, 1)
            self.logs.append(entry)

metrics = MetricsLogger()

trainer = DPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
    callbacks=[metrics],
)

print("Training started...")
trainer.train()
print("Training complete!")

## Step 7: Visualize Training Metrics

Let's see how the loss decreased and how reward margins (chosen - rejected) improved during training.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('dark_background')

logs = metrics.logs
loss_logs = [l for l in logs if 'loss' in l]
reward_logs = [l for l in logs if 'rewards/margins' in l]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
if loss_logs:
    axes[0].plot([l['step'] for l in loss_logs], [l['loss'] for l in loss_logs],
                 color='#818cf8', linewidth=2, marker='o', markersize=4)
    axes[0].set_title('DPO Training Loss', fontsize=14, color='#a78bfa')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

# Reward margins
if reward_logs:
    steps = [l['step'] for l in reward_logs]
    axes[1].plot(steps, [l.get('rewards/chosen', 0) for l in reward_logs],
                 color='#34d399', linewidth=2, marker='o', markersize=4, label='Chosen')
    axes[1].plot(steps, [l.get('rewards/rejected', 0) for l in reward_logs],
                 color='#f87171', linewidth=2, marker='o', markersize=4, label='Rejected')
    axes[1].plot(steps, [l['rewards/margins'] for l in reward_logs],
                 color='#818cf8', linewidth=2, linestyle='--', marker='s', markersize=4, label='Margin')
    axes[1].set_title('Reward Scores', fontsize=14, color='#a78bfa')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Reward')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print summary
if loss_logs:
    print("Loss: %.4f -> %.4f" % (loss_logs[0]['loss'], loss_logs[-1]['loss']))
if reward_logs:
    print("Margin: %.4f -> %.4f" % (reward_logs[0]['rewards/margins'], reward_logs[-1]['rewards/margins']))
    print("(positive margin = model prefers chosen over rejected)")

## Step 8: Before vs After DPO Comparison

The moment of truth — let's compare the model's responses before and after alignment.

In [ ]:
print("=" * 70)
print("BEFORE vs AFTER DPO")
print("=" * 70)

for p in TEST_PROMPTS:
    after = generate(model, p)
    print(f"\n📝 Prompt: {p}")
    print(f"  ❌ BEFORE: {before_responses[p]}")
    print(f"  ✅ AFTER:  {after}")
    print("-" * 70)

## Step 9: Visualize Preference Data & Results

In [ ]:
from IPython.display import HTML, display
import html as html_lib

def card(prompt, chosen, rejected):
    p = html_lib.escape(prompt)
    c = html_lib.escape(chosen)
    r = html_lib.escape(rejected)
    return (
        '<div style="background:#131927;border:1px solid #2a2f45;border-radius:12px;padding:16px;margin:8px 0;font-family:sans-serif;">'
        '  <div style="color:#c4b5fd;font-weight:bold;margin-bottom:10px;">"' + p + '"</div>'
        '  <div style="display:grid;grid-template-columns:1fr 1fr;gap:10px;">'
        '    <div style="background:rgba(52,211,153,0.08);border:1px solid rgba(52,211,153,0.3);border-radius:8px;padding:12px;">'
        '      <div style="color:#34d399;font-size:0.75em;font-weight:bold;text-transform:uppercase;margin-bottom:6px;">Chosen</div>'
        '      <div style="color:#b0b8c8;font-size:0.9em;">' + c + '</div>'
        '    </div>'
        '    <div style="background:rgba(248,113,113,0.08);border:1px solid rgba(248,113,113,0.3);border-radius:8px;padding:12px;">'
        '      <div style="color:#f87171;font-size:0.75em;font-weight:bold;text-transform:uppercase;margin-bottom:6px;">Rejected</div>'
        '      <div style="color:#b0b8c8;font-size:0.9em;">' + r + '</div>'
        '    </div>'
        '  </div>'
        '</div>'
    )

html_out = '<h3 style="color:#a78bfa;font-family:sans-serif;">Training Preference Pairs</h3>'
for ex in PREFERENCE_DATA:
    html_out += card(ex["prompt"], ex["chosen"], ex["rejected"])

display(HTML(html_out))

## Step 10: Save the Aligned Model

In [ ]:
trainer.save_model("./dpo_output/final")
tokenizer.save_pretrained("./dpo_output/final")
print("✅ Model saved to ./dpo_output/final")
print("\nTo load later:")
print('  from peft import AutoPeftModelForCausalLM')
print('  model = AutoPeftModelForCausalLM.from_pretrained("./dpo_output/final")')

---

## Summary: What Just Happened

| Step | What | Why |
|------|------|-----|
| Load model | TinyLlama 1.1B | Small enough for free GPU |
| Preference data | 5 chosen/rejected pairs | Teaches what "good" looks like |
| LoRA | Tiny adapters on attention | Train ~1% of params, not all 1.1B |
| DPO training | β=0.1, 3 epochs | Directly optimize for preferences |
| Result | Model prefers detailed, helpful responses | No reward model or RL needed! |

### Key Takeaway
DPO turns the complex RLHF pipeline into **one simple training loop**. Same result, fraction of the complexity. This is why companies like Anthropic, Meta, and others use DPO variants for alignment.